# Revenue Funnel Analysis - Dorje Teas

[SYNTHETIC DATA - Illustrative only. No internal Dorje data used.]

Purpose: analyze the sample D2C funnel from paid sessions through purchase, then connect revenue movement to founder decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

DATA_DIR = Path('../04_data_model')
if not DATA_DIR.exists():
    DATA_DIR = Path('04_data_model')
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR.resolve()}'

def rupee(value):
    return f'Rs. {value:,.0f}'

orders = pd.read_csv(DATA_DIR / 'sample_orders.csv')
ad_spend = pd.read_csv(DATA_DIR / 'sample_ad_spend.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'], dayfirst=True)
orders['is_discounted'] = orders['discount_amount'] > 0
orders['cm_pct_calc'] = orders['contribution_margin'] / orders['net_revenue'] * 100
orders.head()

## Data Quality Checks

In [ ]:
required_order_cols = {'order_id','order_date','month','customer_id','customer_type','product_category','net_revenue','discount_amount','contribution_margin','cm_pct','channel'}
required_ad_cols = {'week','month','campaign_id','spend','impressions','clicks','sessions','add_to_cart','checkouts','orders','revenue','ctr_pct','cvr_pct','contribution_roas'}
missing_orders = required_order_cols - set(orders.columns)
missing_ad = required_ad_cols - set(ad_spend.columns)
assert not missing_orders, missing_orders
assert not missing_ad, missing_ad
assert len(orders) > 0 and len(ad_spend) > 0
assert (orders['net_revenue'] > 0).all()
quality_summary = pd.DataFrame({
    'table': ['sample_orders', 'sample_ad_spend'],
    'rows': [len(orders), len(ad_spend)],
    'missing_required_columns': [len(missing_orders), len(missing_ad)]
})
quality_summary

## Revenue Overview

In [ ]:
revenue_overview = pd.Series({
    'orders': len(orders),
    'net_revenue': orders['net_revenue'].sum(),
    'gross_revenue': orders['gross_revenue'].sum(),
    'contribution_margin': orders['contribution_margin'].sum(),
    'blended_cm_pct': orders['contribution_margin'].sum() / orders['net_revenue'].sum() * 100,
    'aov': orders['net_revenue'].mean(),
    'discount_rate_pct': orders['discount_amount'].sum() / orders['gross_revenue'].sum() * 100
})
revenue_overview.to_frame('value')

## Monthly Revenue Trend

In [ ]:
monthly_revenue = orders.groupby('month', as_index=False).agg(net_revenue=('net_revenue','sum'), orders=('order_id','count'))
ax = sns.lineplot(data=monthly_revenue, x='month', y='net_revenue', marker='o')
ax.set_title('Dorje Teas Monthly Net Revenue [SYNTHETIC DATA]')
ax.set_xlabel('Month')
ax.set_ylabel('Net revenue (Rs.)')
plt.xticks(rotation=45)
for _, row in monthly_revenue.iterrows():
    ax.text(row['month'], row['net_revenue'], f"{row['net_revenue']/1000:.0f}k", ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## New vs Returning Customer Revenue Split

In [ ]:
customer_split = orders.groupby(['month','customer_type'], as_index=False)['net_revenue'].sum()
fig, ax = plt.subplots(figsize=(12,6))
sns.barplot(data=customer_split, x='month', y='net_revenue', hue='customer_type', ax=ax)
ax.set_title('Dorje Teas New vs Returning Revenue Split [SYNTHETIC DATA]')
ax.set_xlabel('Month')
ax.set_ylabel('Net revenue (Rs.)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
customer_mix = orders.groupby('customer_type').agg(revenue=('net_revenue','sum'), orders=('order_id','count'))
customer_mix['revenue_share_pct'] = customer_mix['revenue'] / customer_mix['revenue'].sum() * 100
customer_mix

## Sessions to Purchase Funnel

In [ ]:
funnel_totals = ad_spend[['impressions','clicks','sessions','add_to_cart','checkouts','orders']].sum()
funnel = funnel_totals.rename_axis('stage').reset_index(name='count')
funnel['drop_from_prior_pct'] = funnel['count'].pct_change().mul(-100).fillna(0)
funnel

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
sns.barplot(data=funnel, x='stage', y='count', color='#4C78A8', ax=ax)
ax.set_title('Dorje Teas Funnel Volume [SYNTHETIC DATA]')
ax.set_xlabel('Funnel stage')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f"{p.get_height():,.0f}", (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## Funnel Drop-Off Percentages

In [ ]:
drop = funnel.loc[funnel['stage'] != 'impressions', ['stage','drop_from_prior_pct']]
ax = sns.barplot(data=drop, x='stage', y='drop_from_prior_pct', color='#F58518')
ax.set_title('Dorje Teas Funnel Drop-Off by Stage [SYNTHETIC DATA]')
ax.set_xlabel('Stage')
ax.set_ylabel('Drop from prior stage (%)')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## CVR by Channel

In [ ]:
channel_cvr = orders.groupby('channel', as_index=False).agg(orders=('order_id','count'), revenue=('net_revenue','sum'))
ad_channel = ad_spend.assign(channel=np.where(ad_spend['platform'].eq('google'),'google_search','meta')).groupby('channel', as_index=False).agg(sessions=('sessions','sum'))
channel_cvr = channel_cvr.merge(ad_channel, on='channel', how='left')
channel_cvr['sessions'] = channel_cvr['sessions'].fillna(channel_cvr['orders'] / 0.04)
channel_cvr['cvr_pct'] = channel_cvr['orders'] / channel_cvr['sessions'] * 100
ax = sns.barplot(data=channel_cvr.sort_values('cvr_pct', ascending=False), x='channel', y='cvr_pct', color='#54A24B')
ax.set_title('Dorje Teas CVR by Channel [SYNTHETIC DATA]')
ax.set_xlabel('Channel')
ax.set_ylabel('CVR (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
channel_cvr

## AOV Views

In [ ]:
aov_category = orders.groupby('product_category', as_index=False).agg(aov=('net_revenue','mean'), orders=('order_id','count')).sort_values('aov', ascending=False)
aov_customer = orders.groupby('customer_type', as_index=False).agg(aov=('net_revenue','mean'), orders=('order_id','count'))
aov_channel = orders.groupby('channel', as_index=False).agg(aov=('net_revenue','mean'), orders=('order_id','count')).sort_values('aov', ascending=False)
fig, axes = plt.subplots(1, 3, figsize=(18,5))
sns.barplot(data=aov_category, x='product_category', y='aov', ax=axes[0], color='#4C78A8')
sns.barplot(data=aov_customer, x='customer_type', y='aov', ax=axes[1], color='#72B7B2')
sns.barplot(data=aov_channel, x='channel', y='aov', ax=axes[2], color='#F58518')
for ax in axes:
    ax.set_ylabel('AOV (Rs.)')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_title('AOV by Product Category')
axes[1].set_title('AOV by Customer Type')
axes[2].set_title('AOV by Channel')
fig.suptitle('Dorje Teas AOV Diagnostics [SYNTHETIC DATA]')
plt.tight_layout()
plt.show()
aov_category

## Product Category Revenue Mix Over Time

In [ ]:
category_mix = orders.pivot_table(index='month', columns='product_category', values='net_revenue', aggfunc='sum', fill_value=0)
category_share = category_mix.div(category_mix.sum(axis=1), axis=0) * 100
ax = category_share.plot(kind='area', stacked=True, figsize=(12,6), colormap='tab20')
ax.set_title('Dorje Teas Product Category Revenue Mix Over Time [SYNTHETIC DATA]')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue share (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Discount Rate by Channel and Category

In [ ]:
discount_channel = orders.groupby('channel').apply(lambda x: x['discount_amount'].sum()/x['gross_revenue'].sum()*100, include_groups=False).reset_index(name='discount_rate_pct')
discount_category = orders.groupby('product_category').apply(lambda x: x['discount_amount'].sum()/x['gross_revenue'].sum()*100, include_groups=False).reset_index(name='discount_rate_pct')
fig, axes = plt.subplots(1, 2, figsize=(16,5))
sns.barplot(data=discount_channel.sort_values('discount_rate_pct', ascending=False), x='channel', y='discount_rate_pct', ax=axes[0], color='#E45756')
sns.barplot(data=discount_category.sort_values('discount_rate_pct', ascending=False), x='product_category', y='discount_rate_pct', ax=axes[1], color='#E45756')
axes[0].set_title('Discount Rate by Channel')
axes[1].set_title('Discount Rate by Product Category')
for ax in axes:
    ax.set_ylabel('Discount rate (%)')
    ax.tick_params(axis='x', rotation=45)
fig.suptitle('Dorje Teas Discount Diagnostics [SYNTHETIC DATA]')
plt.tight_layout()
plt.show()

## Discounted vs Full-Price Contribution Margin

In [ ]:
discount_cm = orders.groupby('is_discounted', as_index=False).agg(orders=('order_id','count'), net_revenue=('net_revenue','sum'), contribution_margin=('contribution_margin','sum'))
discount_cm['cm_pct'] = discount_cm['contribution_margin'] / discount_cm['net_revenue'] * 100
discount_cm['price_status'] = np.where(discount_cm['is_discounted'], 'Discounted', 'Full price')
ax = sns.barplot(data=discount_cm, x='price_status', y='cm_pct', color='#B279A2')
ax.set_title('Dorje Teas Discounted vs Full-Price CM% [SYNTHETIC DATA]')
ax.set_xlabel('Price status')
ax.set_ylabel('Contribution margin (%)')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()
discount_cm

## Key Findings

- Returning customer revenue is the signal to watch because the north star is contribution-margin-positive repeat D2C revenue.
- Strong CTR with weak CVR should trigger landing page or product page diagnosis, not more spend.
- High AOV is useful only if conversion and contribution margin hold.
- Discount-led order growth should not scale when CM% compresses.

## Operator Decisions This Analysis Supports

- If CTR is strong but CVR is weak, fix the landing page, product proof, or offer before increasing traffic.
- If AOV rises while CVR falls, review bundle friction and price-per-cup framing.
- If discounts lift orders but CM% falls, hold discount-led growth and test non-discount trust modules.
- If returning customer revenue is strong, expand product-specific retention flows.

## What Real Dorje Data Would Improve

- Actual Shopify sessions and product page views would make stage-level CVR cleaner.
- Actual campaign attribution would separate Meta, Google Search, email, organic, and direct more precisely.
- Actual discount-code usage would show whether discounts are planned, leaked, or retention-driven.